### Retrieving the results of the training to make predictions on the test dataset and display the confusion matrix an the interpretation tools

In [1]:
from transformers import RobertaTokenizer, RobertaForSequenceClassification, DataCollatorWithPadding
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import numpy as np
import datasets
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import shap
import os


ModuleNotFoundError: No module named 'shap'

In [2]:
labels      = [ "Self-direction: thought", "Self-direction: action", "Stimulation",  "Hedonism", "Achievement", "Power: dominance", "Power: resources", "Face", "Security: personal", "Security: societal", "Tradition", "Conformity: rules", "Conformity: interpersonal", "Humility", "Benevolence: caring", "Benevolence: dependability", "Universalism: concern", "Universalism: nature", "Universalism: tolerance", "No Value"]
num_labels  = len(labels)

train_dataset       = datasets.load_from_disk("../datasets/processed_train")
validation_dataset  = datasets.load_from_disk("../datasets/processed_validation")
test_dataset        = datasets.load_from_disk("../datasets/processed_test")

In [ ]:
model       = RobertaForSequenceClassification.from_pretrained("./results", num_labels = num_labels)
tokenizer   = RobertaTokenizer.from_pretrained("./results")

In [ ]:
model.eval()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

### Confusion matrix :

In [ ]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

def dataset_to_dataloader(encoded_dataset, batch_size=8):
    return DataLoader(encoded_dataset, batch_size=batch_size, collate_fn=data_collator)

test_dataloader = dataset_to_dataloader(test_dataset)

all_logits = []
all_true_labels = []

for batch in test_dataloader:
    input_ids = batch["input_ids"].to(device)
    attention_mask = batch["attention_mask"].to(device)
    labels = torch.tensor(batch["labels"]).to(device)  # Extract true labels

    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs[0]
        all_logits.append(logits.cpu().numpy())
        all_true_labels.append(labels.cpu().numpy())

# Concatenate logits and true labels
all_logits = np.concatenate(all_logits, axis=0)
all_true_labels = np.concatenate(all_true_labels, axis=0)

In [ ]:
# Convert logits to probabilities
probs = torch.sigmoid(torch.tensor(all_logits))

threshold = 0.5  # or any other chosen threshold
predicted_labels = (probs >= threshold).int().numpy()

num_labels = all_true_labels.shape[1]
global_conf_matrix = np.zeros((num_labels, num_labels), dtype=int)

# Compute confusion matrix per label (TP, FP, TN, FN)
for label_idx in range(num_labels):
    # Get true and predicted values for the current label
    true_label = all_true_labels[:, label_idx]
    pred_label = predicted_labels[:, label_idx]
    
    # Compute confusion matrix for the current label
    # cm = confusion_matrix(true_label, pred_label)
    tp = ((true_label == 1) & (pred_label == 1)).sum()  # True positives
    fp = ((true_label == 0) & (pred_label == 1)).sum()  # False positives
    tn = ((true_label == 0) & (pred_label == 0)).sum()  # True negatives
    fn = ((true_label == 1) & (pred_label == 0)).sum()  # False negatives
    
    # Store the confusion matrix (TP, FP, TN, FN) for the current label
    global_conf_matrix[label_idx, label_idx] = tp  # Diagonal represents true positives
    global_conf_matrix[label_idx, (label_idx + 1) % num_labels] = fp  # Store FP
    global_conf_matrix[(label_idx + 1) % num_labels, label_idx] = fn  # Store FN
    global_conf_matrix[(label_idx + 1) % num_labels, (label_idx + 1) % num_labels] = tn

In [ ]:
plt.figure(figsize=(10,10))
sns.heatmap(global_conf_matrix, xticklabels=labels, yticklabels=labels, annot=True, fmt='d', cmap="Blues")
plt.xlabel('Predicted Labels')
plt.ylabel('True Labels')
plt.title('Global Confusion Matrix')
plt.show()

### Explainability :

In [11]:
explainer = shap.Explainer(model)

NameError: name 'shap' is not defined

In [ ]:
# que mettre comme chemin pour le file des labels ???
def load_dataset(directory, load_labels=True):
    sentences_file_path = os.path.join(directory, "sentences.tsv")
    labels_file_path = os.path.join(directory, f"{dataset}.tsv")
    
    data_frame = pd.read_csv(sentences_file_path, encoding="utf-8", sep="\t", header=0)

    if load_labels and os.path.isfile(labels_file_path):
        labels_frame = pd.read_csv(labels_file_path, encoding="utf-8", sep="\t", header=0)
        labels_frame = pd.merge(data_frame, labels_frame, on=["Text-ID", "Sentence-ID"], how="inner")
       
    
    return labels_frame

In [ ]:
directory_train="../datasets/valueeval24/training-english"
data = load_dataset(directory_train)

Visualise the impact on all the output classes

In [ ]:
shap_values = explainer(data["Text"][:3])
shap.plots.text(shap_values)